# Metadata

> Metadata introspection for the Demucs plugin used by cjm-ctl to generate the registration manifest.

In [ ]:
#| default_exp meta

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os
import sys
from typing import Any, Dict
from cjm_media_plugin_demucs import __version__

In [ ]:
#| export
def get_plugin_metadata() -> Dict[str, Any]:  # Plugin metadata for manifest generation
    """Return metadata required to register this plugin with the PluginManager."""
    # Fallback base path (current behavior for backward compatibility)
    base_path = os.path.dirname(os.path.dirname(sys.executable))
    
    # Use CJM config if available, else fallback to env-relative paths
    cjm_data_dir = os.environ.get("CJM_DATA_DIR")
    cjm_models_dir = os.environ.get("CJM_MODELS_DIR")
    
    # Plugin data directory
    plugin_name = "cjm-media-plugin-demucs"
    if cjm_data_dir:
        data_dir = os.path.join(cjm_data_dir, plugin_name)
    else:
        data_dir = os.path.join(base_path, "data")
    
    db_path = os.path.join(data_dir, "demucs_jobs.db")
    
    # Ensure data directory exists
    os.makedirs(data_dir, exist_ok=True)
    
    # Demucs downloads models via torch.hub — TORCH_HOME controls cache location
    if cjm_models_dir:
        torch_home = os.path.join(cjm_models_dir, "torch")
    else:
        torch_home = os.path.join(base_path, ".cache", "torch")
    
    return {
        "name": plugin_name,
        "version": __version__,
        "type": "media-processing",
        "category": "media",
        "interface": "cjm_media_plugin_system.processing_interface.MediaProcessingPlugin",
        
        "module": "cjm_media_plugin_demucs.plugin",
        "class": "DemucsProcessingPlugin",
        
        # Critical: The absolute path to THIS environment's python
        "python_path": sys.executable,
        
        "db_path": db_path,
        
        "resources": {
            "requires_gpu": True,
            "min_gpu_vram_mb": 2048,
            "recommended_gpu_vram_mb": 4096,
            "min_system_ram_mb": 4096
        },
        
        "env_vars": {
            "CUDA_VISIBLE_DEVICES": "0",
            "TORCH_HOME": torch_home
        }
    }

## Testing

In [ ]:
import json

metadata = get_plugin_metadata()
print(json.dumps(metadata, indent=2))

{
  "name": "cjm-media-plugin-demucs",
  "version": "0.0.1",
  "type": "media-processing",
  "category": "media",
  "interface": "cjm_media_plugin_system.processing_interface.MediaProcessingPlugin",
  "module": "cjm_media_plugin_demucs.plugin",
  "class": "DemucsProcessingPlugin",
  "python_path": "/home/innom-dt/miniforge3/envs/cjm-media-plugin-demucs/bin/python3.12",
  "db_path": "/home/innom-dt/miniforge3/envs/cjm-media-plugin-demucs/data/demucs_jobs.db",
  "resources": {
    "requires_gpu": true,
    "min_gpu_vram_mb": 2048,
    "recommended_gpu_vram_mb": 4096,
    "min_system_ram_mb": 4096
  },
  "env_vars": {
    "CUDA_VISIBLE_DEVICES": "0",
    "TORCH_HOME": "/home/innom-dt/miniforge3/envs/cjm-media-plugin-demucs/.cache/torch"
  }
}


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()